# Notebook 03 — LoRA Fine-Tuning
**Project**: IndicSenti — Multilingual Indian Sentiment Analysis  
Trains `ai4bharat/indic-bert` with LoRA (r=16, α=32) using Weights & Biases tracking.

In [ ]:
import os, sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, EarlyStoppingCallback)
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import f1_score, accuracy_score
import wandb

PALETTE = {
    'mint': '#F1F6F4', 'gold': '#FFC801', 'teal': '#114C5A',
    'sage': '#D9E8E2', 'orange': '#FF9932', 'dark': '#172B36',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['mint'], 'axes.facecolor': PALETTE['mint'],
    'axes.titlecolor': PALETTE['teal'], 'figure.dpi': 120,
})

# Config
MODEL_ID    = 'ai4bharat/indic-bert'
OUTPUT_DIR  = Path('../models/indicsenti-lora')
DATA_DIR    = Path('../data')
MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS      = 5
LR          = 2e-4
SEED        = 42
LORA_R      = 16
LORA_ALPHA  = 32
NUM_LABELS  = 3

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# ── W&B initialization ──────────────────────────────────────
wandb.init(
    project='multilingual-sentiment',
    name=f'indicsenti-lora-r{LORA_R}',
    config={
        'model': MODEL_ID, 'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA,
        'batch_size': BATCH_SIZE, 'lr': LR, 'epochs': EPOCHS,
        'max_len': MAX_LEN, 'seed': SEED, 'loss': 'focal',
    },
    tags=['lora', 'indic-bert', 'sentiment', 'multilingual'],
)
print('W&B run:', wandb.run.url)

In [ ]:
# ── Load & tokenize dataset ──────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def load_split(split):
    df = pd.read_csv(DATA_DIR / f'{split}.csv')
    return Dataset.from_pandas(df[['text','label']].rename(columns={'label':'labels'}))

raw = DatasetDict(train=load_split('train'), val=load_split('val'), test=load_split('test'))

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

tokenized = raw.map(tokenize, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print('Tokenized:', tokenized)

In [ ]:
# ── Build LoRA model ─────────────────────────────────────────
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['query', 'value'],
    lora_dropout=0.1, bias='none',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Focal Loss Trainer ───────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = torch.tensor(alpha) if alpha else None
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        focal_weight = (1 - p_t) ** self.gamma
        if self.alpha is not None:
            self.alpha = self.alpha.to(logits.device)
            focal_weight = focal_weight * self.alpha[targets]
        loss = focal_weight * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()

focal_loss = FocalLoss(alpha=[0.40, 0.35, 0.25], gamma=2.0)

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1':  f1_score(labels, preds, average='macro'),
        'accuracy':  accuracy_score(labels, preds),
        'f1_pos':    f1_score(labels, preds, average=None)[0],
        'f1_neu':    f1_score(labels, preds, average=None)[1],
        'f1_neg':    f1_score(labels, preds, average=None)[2],
    }

print('FocalLossTrainer ready.')

In [ ]:
# ── Training arguments ───────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=4,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='wandb',
    logging_steps=50,
    seed=SEED,
    dataloader_num_workers=4,
)

trainer = FocalLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Trainer configured.')

In [ ]:
# ── Train ────────────────────────────────────────────────────
train_result = trainer.train()
print('Training complete.')
print(f"  Total steps:  {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────
test_results = trainer.evaluate(tokenized['test'])
print('Test Results:')
for k, v in test_results.items():
    if isinstance(v, float): print(f'  {k}: {v:.4f}')

wandb.log({'test/macro_f1': test_results.get('eval_macro_f1', 0),
           'test/accuracy': test_results.get('eval_accuracy', 0)})

In [ ]:
# ── Fig 8: Training curves ────────────────────────────────────
log_history = trainer.state.log_history
train_losses = [(l['epoch'], l['loss']) for l in log_history if 'loss' in l and 'eval_loss' not in l]
val_metrics  = [(l['epoch'], l['eval_macro_f1'], l['eval_loss']) for l in log_history
                if 'eval_macro_f1' in l]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Figure 8 — Training Curves', color=PALETTE['teal'], fontweight='bold')

# Loss
if train_losses:
    te, tl = zip(*train_losses)
    axes[0].plot(te, tl, color=PALETTE['teal'], linewidth=2, label='Train Loss')
if val_metrics:
    ve, vf, vl = zip(*val_metrics)
    axes[0].plot(ve, vl, color=PALETTE['orange'], linewidth=2, linestyle='--', label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# F1
if val_metrics:
    axes[1].plot(ve, vf, color=PALETTE['gold'], linewidth=2.5, marker='o', label='Val Macro-F1')
    axes[1].axhline(max(vf), color=PALETTE['teal'], linestyle=':', linewidth=1.5,
                    label=f'Best: {max(vf):.3f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro-F1')
axes[1].set_title('Validation Macro-F1')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/fig8_training_curves.png', bbox_inches='tight')
plt.show()
wandb.log({'training_curves': wandb.Image('../reports/fig8_training_curves.png')})

In [ ]:
wandb.finish()
print('Run complete. Model saved to:', OUTPUT_DIR)